In [9]:
from dotenv import load_dotenv
load_dotenv()
import json


In [2]:
from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI()

In [3]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [4]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [5]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [6]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [7]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [10]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Ollama run locally install local model FAQ"}
iteration #2...
function_call: search {"query":"Ollama local server ollama serve localhost 11434 FAQ connection refused"}
iteration #3...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: install the `.pkg`
     - **Windows**: install the `.msi`
     - **Linux**:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a local model**
   ```bash
   ollama run llama3
   ```
   This will download the model, start it locally, and open a chat-style prompt.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should see a response indicating the server is up.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages

'To run Ollama locally:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - Choose your OS:\n     - **macOS**: install the `.pkg`\n     - **Windows**: install the `.msi`\n     - **Linux**:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Start a local model**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model, start it locally, and open a chat-style prompt.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should see a response indicating the server is up.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error, restart the server with:\n\n```bash\nnohup ollama serve

In [11]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late discovered the course can I still join enrollment deadline late registration"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

One important caveat: if you want to receive a certificate, you need to submit your project while submissions are still being accepted. Also, certificates are only available for the live cohort, not self-paced participation.

If you want, I can also help with how to start from where you are now or explain the certificate requirements.


'Yes — you can still join the course.\n\nOne important caveat: if you want to receive a certificate, you need to submit your project while submissions are still being accepted. Also, certificates are only available for the live cohort, not self-paced participation.\n\nIf you want, I can also help with how to start from where you are now or explain the certificate requirements.'

In [12]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment late join access FAQ"}
iteration #2...
function_call: search {"query":"certificate project still accepting submissions join course self-paced live cohort peer-review course running FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join.

If you want a certificate, make sure to submit your project while submissions are still being accepted. The course also has a live-cohort requirement for certificates, so self-paced participation alone doesn’t qualify.

If you want, I can also help you with how to start the course or explain the certificate requirements in more detail.


'Yes — you can still join.\n\nIf you want a certificate, make sure to submit your project while submissions are still being accepted. The course also has a live-cohort requirement for certificates, so self-paced participation alone doesn’t qualify.\n\nIf you want, I can also help you with how to start the course or explain the certificate requirements in more detail.'

In [13]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess definition opening why play queen gambit"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a classic **chess opening**.

It starts with:
1. **d4 d5**
2. **c4**

White offers the **c-pawn** to try to draw Black’s **d-pawn** away from the center, so White can gain more control of the center squares.

A few quick points:
- It’s called a **gambit** because White may be offering a pawn, though Black doesn’t have to accept it.
- It’s one of the oldest and most respected openings in chess.
- Common ideas are **strong center control**, **developing pieces**, and **creating pressure** on Black’s position.

If you want, I can also explain:
- the **main variations** of the Queen’s Gambit,
- the difference between **Queen’s Gambit Accepted** and **Declined**,
- or show a **simple move-by-move example**.


'The **Queen’s Gambit** is a classic **chess opening**.\n\nIt starts with:\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c-pawn** to try to draw Black’s **d-pawn** away from the center, so White can gain more control of the center squares.\n\nA few quick points:\n- It’s called a **gambit** because White may be offering a pawn, though Black doesn’t have to accept it.\n- It’s one of the oldest and most respected openings in chess.\n- Common ideas are **strong center control**, **developing pieces**, and **creating pressure** on Black’s position.\n\nIf you want, I can also explain:\n- the **main variations** of the Queen’s Gambit,\n- the difference between **Queen’s Gambit Accepted** and **Declined**,\n- or show a **simple move-by-move example**.'

In [14]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening queen gambit course FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find any FAQ entry for “queen gambit,” and it looks like that’s not a course/logistics question.

If you meant “Queen’s Gambit” as the chess opening, that isn’t covered in the course FAQ I have access to. If you have a course-related question, send it over and I’ll look it up.

Are there other areas you want to explore?


'I couldn’t find any FAQ entry for “queen gambit,” and it looks like that’s not a course/logistics question.\n\nIf you meant “Queen’s Gambit” as the chess opening, that isn’t covered in the course FAQ I have access to. If you have a course-related question, send it over and I’ll look it up.\n\nAre there other areas you want to explore?'